# 📈 Notebook 06 — Evaluation

---

## Project Context

| Field | Detail |
|---|---|
| **Competition** | SATRIA DATA 2026 — Big Data Challenge |
| **Experiment** | Baseline CNN — Experiment 01 |
| **Stage** | Evaluation (Step 4 of 7) |
| **Primary Metric** | Macro-averaged F1 Score |

---

## Tujuan Tahap Evaluasi

Memvalidasi kemampuan generalisasi model terhadap Validation Set menggunakan `src/evaluation/metrics.py` dan `src/evaluation/visualizer.py`.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & SETUP
# ============================================================
import sys
import logging
import json
from pathlib import Path
import yaml
import torch

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.preprocessing.splitter import collect_dataset, stratified_split
from src.preprocessing.dataloader import build_val_loader
from src.models.cnn.baseline import build_baseline_cnn
from src.evaluation.metrics import run_inference, compute_metrics, get_classification_report
from src.evaluation.visualizer import EvaluatorVisualizer

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

with open(PROJECT_ROOT / 'configs' / 'baseline.yaml', 'r') as f:
    config = yaml.safe_load(f)

set_seed(config['experiment']['seed'])

---
## Step 1 — Load Validation Data & Model Weights
Memuat model terbaik berdasarkan performa saat training berlangsung.

In [ ]:
# ============================================================
# CELL 2 — LOAD DATA & MODEL
# ============================================================
data_root = PROJECT_ROOT / config['data']['raw_dir']
train_dir = data_root / config['data']['train_subdir']
class_names = list(config['data']['class_names'].values())
exp_name = config['experiment']['name']

df_all = collect_dataset(train_dir)
_, df_val = stratified_split(df_all, train_ratio=config['split']['train_ratio'], val_ratio=config['split']['val_ratio'], seed=config['experiment']['seed'])

val_loader = build_val_loader(
    df_val=df_val,
    batch_size=config['dataloader']['batch_size'],
    num_workers=config['dataloader']['num_workers'],
    image_size=config['preprocessing']['image_size'],
    resize_to=config['preprocessing']['resize_to'],
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_baseline_cnn(num_classes=len(class_names))

checkpoints_dir = PROJECT_ROOT / config['output']['checkpoints_dir']
model_path = checkpoints_dir / f"{exp_name}_best_model.pth"

model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
logger.info(f"Berhasil memuat model dari {model_path.name}")

---
## Step 2 — Run Inference & Calculate Metrics
Mengekstrak F1-Score, Precision, Recall.

In [ ]:
# ============================================================
# CELL 3 — INFERENCE
# ============================================================
all_preds, all_labels, all_probs = run_inference(model, val_loader, device)

metrics_report = compute_metrics(all_labels, all_preds, class_names)
clf_report_str = get_classification_report(all_labels, all_preds, class_names)

reports_dir = PROJECT_ROOT / config['output']['reports_dir']
with open(reports_dir / f"{exp_name}_evaluation_metrics.json", "w") as f:
    json.dump(metrics_report, f, indent=4)

print("\n" + "="*50)
print("  HASIL EVALUASI")
print("="*50)
print(f"⭐ Macro F1 Score : {metrics_report['macro_f1']:.4f}")
print(f"   Accuracy       : {metrics_report['accuracy']:.4f}")
print("\nDetail per Kelas:")
print(clf_report_str)

---
## Step 3 — Visualizations
Membuat *Confusion Matrix* dan *Error Analysis* untuk membantu interpretasi perilaku model.

In [ ]:
# ============================================================
# CELL 4 — VISUALIZE
# ============================================================
figures_dir = PROJECT_ROOT / config['output']['figures_dir']
visualizer = EvaluatorVisualizer(exp_name, figures_dir, class_names)

visualizer.plot_confusion_matrix(all_labels, all_preds, normalize=True)
visualizer.plot_per_class_metrics(metrics_report)
visualizer.plot_error_analysis(all_labels, all_preds)